In [3]:
import os 
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from typing import Optional

In [4]:
def combine_all_years_to_df(data_dir, years=range(2016, 2024)):
    """
    Function to combine data for multiple years into a single DataFrame.
    
    Args: 
        - years (list of int): List of years to combine.
    Returns:
        - combined_df (pd.DataFrame): A DataFrame containing all combined data for the given years.
    """
    combined = []
    for year in years:
        path = os.path.join(data_dir, f"playbyplay-{str(year)}.json")
        df = pd.read_json(path)
        combined.append(df)
    
    combined_df = pd.concat(combined, ignore_index=True)
    return combined_df

In [5]:
df = combine_all_years_to_df('data')
df

,id,season,gameType,limitedScoring,gameDate,venue,venueLocation,startTimeUTC,easternUTCOffset,venueUTCOffset,...,otInUse,clock,displayPeriod,maxPeriods,gameOutcome,plays,rosterSpots,regPeriods,summary,specialEvent
0,2016020001,20162017,2,False,2016-10-12,{'default': 'Canadian Tire Centre'},{'default': 'Ottawa'},2016-10-12T23:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,"{'lastPeriodType': 'OT', 'otPeriods': 1}","[{'eventId': 5, 'periodDescriptor': {'number':...","[{'teamId': 9, 'playerId': 8467493, 'firstName...",3,{},NaN
1,2016020002,20162017,2,False,2016-10-12,{'default': 'United Center'},{'default': 'Chicago'},2016-10-13T00:00:00Z,-04:00,-05:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 5, 'periodDescriptor': {'number':...","[{'teamId': 16, 'playerId': 8466148, 'firstNam...",3,{},NaN
2,2016020003,20162017,2,False,2016-10-12,{'default': 'Rogers Place'},{'default': 'Edmonton'},2016-10-13T02:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 20, 'playerId': 8468674, 'firstNam...",3,{},NaN
3,2016020004,20162017,2,False,2016-10-12,{'default': 'SAP Center at San Jose'},{'default': 'San Jose'},2016-10-13T02:30:00Z,-04:00,-07:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 5, 'periodDescriptor': {'number':...","[{'teamId': 28, 'playerId': 8466138, 'firstNam...",3,{},NaN
4,2016020005,20162017,2,False,2016-10-13,{'default': 'KeyBank Center'},{'default': 'Buffalo'},2016-10-13T23:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 7, 'playerId': 8467407, 'firstName...",3,{},NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10378,2023030413,20232024,3,False,2024-06-13,{'default': 'Rogers Place'},{'default': 'Edmonton'},2024-06-14T00:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 102, 'periodDescriptor': {'number...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN
10379,2023030414,20232024,3,False,2024-06-15,{'default': 'Rogers Place'},{'default': 'Edmonton'},2024-06-16T00:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 52, 'periodDescriptor': {'number'...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN
10380,2023030415,20232024,3,False,2024-06-18,{'default': 'Amerant Bank Arena'},{'default': 'Sunrise'},2024-06-19T00:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 102, 'periodDescriptor': {'number...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN
10381,2023030416,20232024,3,False,2024-06-21,{'default': 'Rogers Place'},{'default': 'Edmonton'},2024-06-22T00:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 52, 'periodDescriptor': {'number'...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN


In [6]:
class PlayByPlayViewer:

    
    def __init__(self, data_dir: str, rink_image_path: str = "../nhl_rink.png"):

        self.rink_img = mpimg.imread(rink_image_path)
        
        
        print("Loading data...")
        self.data_dir = data_dir
        raw_df = self._combine_all_years_to_df(self.data_dir)
        
        print("Performing basic data preprocessing...")
        self.df = self._prepare_data(raw_df)
        
        print(f"Loaded {len(self.df)} games")
        
        self.seasons = None
        self.current_plays = None
        self.current_game_row = None 
        
        self._build_player_cache()    
        self._build_direction_cache()  
        self._create_widgets()
        self._setup_handlers()
        
            
    def _combine_all_years_to_df(self, data_dir, years=range(2016, 2024)):
        """
        Function to combine data for multiple years into a single DataFrame.
        
        Args: 
            - years (list of int): List of years to combine.
        Returns:
            - combined_df (pd.DataFrame): A DataFrame containing all combined data for the given years.
        """
        combined = []
        for year in years:
            path = os.path.join(data_dir, f"playbyplay-{str(year)}.json")
            df = pd.read_json(path)
            combined.append(df)
        
        combined_df = pd.concat(combined, ignore_index=True)
        return combined_df
    
        
    def _prepare_data(self, df) -> pd.DataFrame:

        """
        Performs basic preprocessing on the raw DataFrame of games.

        This method ensures that the data is in a consistent format and easy
        to manipulate for other functions in the class. It performs two main operations:
        1. Converts the 'gameDate' column to pandas datetime objects.
        2. Sorts the entire DataFrame in chronological order of the games.

        Args:
            - df (pd.DataFrame): The raw DataFrame loaded from the JSON files,
              containing the data for all games.

        Returns:
            - df (pd.DataFrame): A cleaned and date-sorted DataFrame.

        """
        df['gameDate'] = pd.to_datetime(df['gameDate'])
        df = df.sort_values('gameDate').reset_index(drop=True)
        return df
    
    
    def _build_player_cache(self) -> None:
        """
        Builds an internal cache mapping game IDs to player information extracted 
        from roster data.
        
        Returns:
            - None: Populates self.player_cache as a nested dictionary where keys 
            are game IDs, and values are dictionaries mapping player IDs to their 
            name and team abbreviation.
        """
        
        print("Building player cache...")
        self.player_cache = {}
        
        for i, row in self.df.iterrows():
            game_id = row['id']
            self.player_cache[game_id] = {}
            
            if 'rosterSpots' in row and row['rosterSpots']:
                for spot in row['rosterSpots']:
                    if 'playerId' in spot:
                        player_id = spot['playerId']
                        first_name = spot.get('firstName', {}).get('default', '')
                        last_name = spot.get('lastName', {}).get('default', '')
                        team_id = spot.get('teamId')
                        
                        team_abbrev = 'N/A'
                        if team_id == row['homeTeam'].get('id'):
                            team_abbrev = row['homeTeam'].get('abbrev', 'HOME')
                        elif team_id == row['awayTeam'].get('id'):
                            team_abbrev = row['awayTeam'].get('abbrev', 'AWAY')
                        
                        self.player_cache[game_id][player_id] = {
                            'name': f"{first_name} {last_name}".strip(),
                            'team': team_abbrev}
    
    
    def _create_widgets(self) -> None:
        """
        Initializes all interactive widgets for the game play visualization interface.
            
        Returns:
            - None
        """

        season_options = []
        path = os.path.join(self.data_dir)
        for filename in os.listdir(path):
            year = filename.split('-')[1].split('.')[0]
            season_options.append(year)
                
        self.season_dropdown = widgets.Dropdown(options=sorted(season_options),
                                                description='Season:')
        
        self.game_type_dropdown = widgets.Dropdown(options=[('Select a season first', None)],
                                                   description='Game Type:')
        
        self.date_dropdown = widgets.Dropdown(options=[('Select a game type first', None)],
                                              description='Date:')
        
        self.matchup_dropdown = widgets.Dropdown(options=[('Select a date first', None)],
                                                 description='Matchup:')
        
        self.play_slider = widgets.IntSlider(
            value=0, min=0, max=0, step=1,
            description='Play:',
            disabled=True,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            readout_format='d',
        )
        
        self.play_info = widgets.HTML(value="<i>Select a game to view a timeline of the plays</i>")
        self.output = widgets.Output()

    
    def _setup_handlers(self) -> None:
        """
        Attaches event handlers to widgets to respond to user interactions.
        
        Returns:
            - None
        
        """
        self.season_dropdown.observe(self._on_season_change, names='value')
        self.game_type_dropdown.observe(self._on_game_type_change, names='value')
        self.date_dropdown.observe(self._on_date_change, names='value')
        self.matchup_dropdown.observe(self._on_matchup_change, names='value')
        self.play_slider.observe(self._on_play_change, names='value')
    
    
    def _on_season_change(self, change) -> None:
        """
        Handles the change event for the season selection dropdown.

        Filters the main DataFrame based on the selected season and updates the 
        game type dropdown (e.g., "Regular Season", "Playoffs") with the
        game types available for that season.

        Args:
            - change (dict): A dictionary from ipywidgets containing event details,
                           including the new selected value.
        Returns:
            - None
        """
        
        selected_season = change['new']
        if selected_season is None:
            self.game_type_dropdown.options = [('Select a season first', None)]
            self.game_type_dropdown.disabled = True
            self.date_dropdown.options = [('Select a game type first', None)]
            self.date_dropdown.disabled = True
            self.matchup_dropdown.options = [('Select a date first', None)]
            self.matchup_dropdown.disabled = True
            return
        
        season_games = self.df[self.df['id'].astype(str).str[:4] == selected_season]
        
        if len(season_games) == 0:
            self.game_type_dropdown.options = [('No games in this season', None)]
            self.game_type_dropdown.disabled = True
            return
        
        game_types = []
        
        regular_games = season_games[season_games['id'].astype(str).str[4:6] == '02']
        if len(regular_games) > 0:
            game_types.append(('Regular Season', 'regular'))
        
        playoff_games = season_games[season_games['id'].astype(str).str[4:6] == '03']
        if len(playoff_games) > 0:
            game_types.append(('Playoffs', 'playoffs'))
        
        if game_types:
            self.game_type_dropdown.options = game_types
            self.game_type_dropdown.disabled = False
            
            self.game_type_dropdown.value = game_types[0][1]
            
        else:
            self.game_type_dropdown.options = [('No games found', None)]
            self.game_type_dropdown.disabled = True
    
    
    def _on_game_type_change(self, change) -> None:
        
        """
        Handles the change event for the game type dropdown.

        Based on the selected season and game type, this method filters the games
        and populates the date dropdown with a sorted list of unique game dates.

        Args:
            - change (dict): A dictionary from ipywidgets containing event details.
            
        Returns:
            - None
        """
        
        selected_game_type = change['new']
        selected_season = self.season_dropdown.value
        
        if selected_game_type is None or selected_season is None:
            self.date_dropdown.options = [('Select a game type first', None)]
            self.date_dropdown.disabled = True
            self.matchup_dropdown.options = [('Select a date first', None)]
            self.matchup_dropdown.disabled = True
            return
        
        season_games = self.df[self.df['id'].astype(str).str[:4] == selected_season]
        
        if selected_game_type == 'regular':
            filtered_games = season_games[season_games['id'].astype(str).str[4:6] == '02']
        else:  
            filtered_games = season_games[season_games['id'].astype(str).str[4:6] == '03']
        
        if len(filtered_games) == 0:
            self.date_dropdown.options = [('No games found', None)]
            self.date_dropdown.disabled = True
            return
        
        unique_dates = sorted(filtered_games['gameDate'].dt.date.unique(), reverse=True)
        date_options = [(str(date), date) for date in unique_dates]
        
        self.date_dropdown.options = date_options
        self.date_dropdown.disabled = False
        if date_options:
            self.date_dropdown.value = date_options[0][1]
   
        
    def _on_date_change(self, change) -> None:
        """
        Handles the change event for the date selection dropdown.

        Filters the games for the selected date and updates the matchup dropdown
        with a list of games played on that day, showing teams and scores.

        Args:
           - change (dict): A dictionary from ipywidgets containing event details.
            
        Returns:
           - None
        """
        selected_date = change['new']
        selected_season = self.season_dropdown.value
        selected_game_type = self.game_type_dropdown.value
        
        if selected_date is None or selected_season is None or selected_game_type is None:
            self.matchup_dropdown.options = [('Select a date first', None)]
            self.matchup_dropdown.disabled = True
            return
        
        season_games = self.df[self.df['id'].astype(str).str[:4] == selected_season]
        
        if selected_game_type == 'regular':
            season_games = season_games[season_games['id'].astype(str).str[4:6] == '02']
        else:
            season_games = season_games[season_games['id'].astype(str).str[4:6] == '03']
        
        games_on_date = season_games[season_games['gameDate'].dt.date == selected_date]
        
        if len(games_on_date) == 0:
            self.matchup_dropdown.options = [('No games on this date', None)]
            self.matchup_dropdown.disabled = True
            self.play_slider.disabled = True
            self.play_info.value = "<i>No games found for selected date</i>"
        else:
            matchup_options = []
            for idx, row in games_on_date.iterrows():
                away_team = row['awayTeam']['abbrev']
                home_team = row['homeTeam']['abbrev']
                away_score = row['awayTeam'].get('score', 0)
                home_score = row['homeTeam'].get('score', 0)
                label = f"{away_team} ({away_score}) @ {home_team} ({home_score})"
                matchup_options.append((label, idx))
            
            self.matchup_dropdown.options = matchup_options
            self.matchup_dropdown.disabled = False
            
            if len(matchup_options) > 0:
                self.matchup_dropdown.value = matchup_options[0][1]
        
        
    def _on_matchup_change(self, change) -> None:
        """
        Handles the change event for the matchup selection dropdown.

        Loads the play-by-play data for the selected game, configures the
        play slider's range, and triggers the initial display update.

        Args:
            - change (dict): A dictionary from ipywidgets containing event details.
            
        Returns:
            - None
        """
        game_idx = change['new']
        if game_idx is None:
            self.play_slider.disabled = True
            self.play_info.value = "<i>Select a game to view plays</i>"
            self.current_plays = None
            self.current_game_row = None
            with self.output:
                clear_output()
            return
        
        game_row = self.df.iloc[game_idx]
        if isinstance(game_row['plays'], list):
            plays_df = pd.json_normalize(game_row['plays'])
        else:
            plays_df = game_row['plays']
        
        self.current_plays = plays_df
        self.current_game_row = game_row
        
        if len(plays_df) > 0:
            self.play_slider.max = len(plays_df) - 1
            self.play_slider.value = 0
            self.play_slider.disabled = False
            home_team = game_row['homeTeam']['abbrev']
            away_team = game_row['awayTeam']['abbrev']
            self.play_info.value = f"<b>Game:</b> {away_team} at {home_team} | <b>Number of Total Plays:</b> {len(plays_df)}"
            self._update_display()
        else:
            self.play_slider.disabled = True
            self.play_info.value = "<i>No plays found in this game</i>"


    def _on_play_change(self, change):
        self._update_display()
    
    
    def _get_player_display(self, game_id: str, player_id: str) -> str:
        """
        Retrieves formatted player display name with team abbreviation from the player cache.
        
        Args:
            - game_id: Unique identifier for the game.
            - player_id: Unique identifier for the player.
        
        Returns:
            - str: "Player Name (TEAM)" if found in cache
                   "Player {player_id}" if not found 
                   "N/A" if player_id is None or NaN
        """

        if not player_id or pd.isna(player_id):
            return "N/A"
        player_info = self.player_cache.get(game_id, {}).get(player_id)
        
        if player_info:
            return f"{player_info['name']} ({player_info['team']})"
        return f"Player {player_id}"
    
    
    def _describe_event(self, game_id: int, event) -> str:
        """
        Generates a human-readable description of a game event with player names and details.
        
        Args:
            - game_id: Unique identifier for the game
            - event: Dictionary containing event data including type, player IDs, and details
        
        Returns:
            - str: Formatted description of the event tailored to the event type. 
                   Returns a title-cased version of the event type for unknown event types.
        """
        play = event.get('typeDescKey', 'unknown')
        
        def get_player_id(key):
            val = event.get(f'details.{key}')
            if pd.isna(val) or val is None:
                val = event.get(key)
            return val
        
        if play == 'goal':
            scorer_id = get_player_id('scoringPlayerId')
            assist1_id = get_player_id('assist1PlayerId')
            assist2_id = get_player_id('assist2PlayerId')
            
            desc = f"GOAL by {self._get_player_display(game_id, scorer_id)}"
            assists = []
            if assist1_id and not pd.isna(assist1_id):
                assists.append(self._get_player_display(game_id, assist1_id))
            if assist2_id and not pd.isna(assist2_id):
                assists.append(self._get_player_display(game_id, assist2_id))
            if assists:
                desc += f" (Assisted by: {', '.join(assists)})"
            return desc
        
        elif play == 'shot-on-goal':
            shooter_id = get_player_id('shootingPlayerId')
            goalie_id = get_player_id('goalieInNetId')
            return f"Shot by {self._get_player_display(game_id, shooter_id)} saved by goaltender {self._get_player_display(game_id, goalie_id)}"
        
        elif play == 'blocked-shot':
            shooter_id = get_player_id('shootingPlayerId')
            blocker_id = get_player_id('blockingPlayerId')
            return f"Shot by {self._get_player_display(game_id, shooter_id)} blocked by {self._get_player_display(game_id, blocker_id)}"
        
        elif play == 'missed-shot':
            shooter_id = get_player_id('shootingPlayerId')
            return f"Missed shot by {self._get_player_display(game_id, shooter_id)}"
        
        elif play == 'hit':
            hitter_id = get_player_id('hittingPlayerId')
            hittee_id = get_player_id('hitteePlayerId')
            return f"Hit: {self._get_player_display(game_id, hitter_id)} on {self._get_player_display(game_id, hittee_id)}"
        
        elif play == 'faceoff':
            winner_id = get_player_id('winningPlayerId')
            return f"Faceoff won by {self._get_player_display(game_id, winner_id)}"
        
        elif play == 'penalty':
            return f"Penalty: {event.get('details.descKey', 'Unknown').replace('-', ' ').title()}"

        else:
            return play.replace('-', ' ').title()
        
        
    def _build_direction_cache(self) -> None:
        """
        Builds a map of the game IDs, periods, and team IDs to their attacking 
        direction based on average shot coordinates for a given period.
        
        Returns:
            - None
        """

        print("Building direction cache...")
        self.direction_cache = {}

        for _, row in self.df.iterrows():
            
            game_id = row['id']
            self.direction_cache[game_id] = {}
            
            if not isinstance(row['plays'], list):
                continue
            
            plays_df = pd.json_normalize(row['plays'])
            
            if 'details.xCoord' not in plays_df.columns:
                continue

            shot_like_events = plays_df[plays_df['typeDescKey'].isin(
                ['shot-on-goal', 'missed-shot', 'blocked-shot', 'goal']
            )].copy()

            if shot_like_events.empty:
                continue

            if 'details.eventOwnerTeamId' in shot_like_events.columns:
                team_series = shot_like_events['details.eventOwnerTeamId']
            elif 'eventOwnerTeamId' in shot_like_events.columns:
                team_series = shot_like_events['eventOwnerTeamId']
            else:
                continue

            shot_like_events['teamId'] = team_series

            if 'periodDescriptor.number' in shot_like_events.columns:
                shot_like_events['period'] = shot_like_events['periodDescriptor.number']
            elif 'periodDescriptor' in shot_like_events.columns:
                shot_like_events['period'] = shot_like_events['periodDescriptor'].apply(
                    lambda p: p.get('number') if isinstance(p, dict) else None
                )

            for (team_id, period), subset in shot_like_events.groupby(['teamId', 'period']):
                subset = subset.dropna(subset=['details.xCoord'])
                if subset.empty:
                    continue
                mean_x = subset['details.xCoord'].mean()
                direction = 'left' if mean_x < 0 else 'right'
                self.direction_cache[game_id].setdefault(period, {})[team_id] = direction

    
    def _get_attack_direction(self, game_id: str, event) -> Optional[str]:
        '''
        Retrieves the attacking direction for the team that made the play/event.
        
        Args:
            - game_id: Unique identifier for the game
            - event: Dictionary or Series containing event details including team 
              and period information
        
        Returns:
            - str: Attacking direction ('left' or 'right') for the team's play/event 
                   in the given period
        '''
        
        event_team_id = event.get('details.eventOwnerTeamId') or event.get('eventOwnerTeamId')
        period = event.get('periodDescriptor.number', event.get('periodDescriptor', {}).get('number', 1))
        
        if (game_id in self.direction_cache and
            period in self.direction_cache[game_id] and
            event_team_id in self.direction_cache[game_id][period]):
            return self.direction_cache[game_id][period][event_team_id]
        
        return None
        
        
    def _update_display(self, change=None) -> None:
        '''
        Updates the visualization output to display the current play on the rink 
        with additional details.
    
        Returns:
            - None: Clears and updates self.output with game information, play details, 
                    and a visualization showing the event location on the rink image.         
        '''

        with self.output:
            clear_output(wait=True)
            
            if self.current_game_row is None or self.current_plays is None:
                return
            
            game_row = self.current_game_row
            plays_df = self.current_plays
            
            if len(plays_df) == 0:
                print("No plays in this game")
                return
            
            play_idx = self.play_slider.value
            event = plays_df.iloc[play_idx]
            
            game_id = game_row['id']
            home_team_id = game_row['homeTeam']['id']
            away_team_id = game_row['awayTeam']['id']
            home_team = game_row['homeTeam']['abbrev']
            away_team = game_row['awayTeam']['abbrev']
            
            period = event.get('periodDescriptor.number', event.get('periodDescriptor', {}).get('number', '?'))
            
            home_attack_dir = None
            away_attack_dir = None
            
            if game_id in self.direction_cache and period in self.direction_cache[game_id]:
                home_attack_dir = self.direction_cache[game_id][period].get(home_team_id)
                away_attack_dir = self.direction_cache[game_id][period].get(away_team_id)
            
            
            period = event.get('periodDescriptor.number', event.get('periodDescriptor', {}).get('number', '?'))
            time = event.get('timeInPeriod', 'N/A')
            description = self._describe_event(game_id, event)
            
            
            _, ax = plt.subplots(figsize=(12, 5))
            ax.imshow(self.rink_img, extent=[-100, 100, -42.5, 42.5])
                        
            x = event.get('details.xCoord')
            y = event.get('details.yCoord')
            
            if pd.isna(x):
                x = event.get('xCoord')
            if pd.isna(y):
                y = event.get('yCoord')
            
            if not pd.isna(x) and not pd.isna(y):
                
                ax.scatter(x, y, marker='x', c='fuchsia', s=200, linewidths=2)
                
            if home_attack_dir and away_attack_dir:
                if home_attack_dir == 'right':
                    left_team = home_team
                    right_team = away_team
                else:
                    left_team = away_team
                    right_team = home_team
                
                title = f"{left_team}     {description} | Period {period} @ {time}     {right_team}"
            else:
                title = f"{description} | Period {period} @ {time}"
            
            ax.set_title(title, fontsize=11, fontweight='bold')
            ax.set_xlabel("Feet (x)", fontsize=10)
            ax.set_ylabel("Feet (y)", fontsize=10)
            ax.grid(False)
            plt.tight_layout()
            plt.show()
    
    
    def display(self) -> None: 
        
        controls = widgets.VBox([
            self.season_dropdown,
            self.game_type_dropdown,
            self.date_dropdown,
            self.matchup_dropdown,
            self.play_slider
            ])
        
        display(controls, self.output)
        

In [5]:
viewer = PlayByPlayViewer(data_dir='data')
viewer.display()

Loading data...
Performing basic data preprocessing...
Loaded 10383 games
Building player cache...
Building direction cache...


Output()